# Fine-tune `bge-base-en-v1.5` on the AU-housing triplets

Runs Task 2.10 (`src/training/train_embeddings.py`) on a Colab T4. Expected wall time: **15–30 min**, **~5–7 GB VRAM**.

## Before starting
1. **Runtime → Change runtime type → GPU (T4)**.
2. Have these two files from your local repo ready to upload:
   - `data/training/triplets.jsonl` (3,943 rows)
   - `src/training/train_embeddings.py`

## What this notebook does
1. Confirms the GPU is attached.
2. Installs `sentence-transformers`.
3. Prompts you to upload the two files.
4. Runs the training script (3 epochs, batch 64, lr 2e-5).
5. Zips the output dir and downloads it.

Drop the resulting `bge-au-housing-v1.zip` into `models/` locally and unzip — it's drop-in compatible with `embed.py` / `retriever.py`.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

Colab already has `torch` with CUDA. We only need `sentence-transformers`, which pulls in `transformers` and `accelerate`.

In [ ]:
!pip install -q 'sentence-transformers>=3.0'

## 3. Upload `triplets.jsonl` and `train_embeddings.py`

When the file picker opens, select **both** files together.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, blob in uploaded.items():
    print(f'{name}: {len(blob):,} bytes')

In [ ]:
import os
assert os.path.exists('triplets.jsonl'), 'triplets.jsonl missing — re-run the upload cell'
assert os.path.exists('train_embeddings.py'), 'train_embeddings.py missing — re-run the upload cell'
with open('triplets.jsonl', 'r', encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print(f'triplets.jsonl rows: {n}')

## 4. Train

Args match the recommended Colab T4 recipe in the script's docstring:
- `--batch 64` — fits in T4 16 GB with `bge-base` (~5–7 GB used)
- `--epochs 3` — typical sweet spot for contrastive fine-tuning
- `--lr 2e-5` — standard for sentence-transformers fine-tuning

Training prints loss per step and writes `eval/RerankingEvaluator_dev_results.csv` with MAP/MRR@10 per epoch.

In [ ]:
!python train_embeddings.py \
    --triplets triplets.jsonl \
    --out-dir bge-au-housing-v1 \
    --batch 64 --epochs 3 --lr 2e-5

## 5. Inspect dev metrics

The training run distils the evaluator CSV into `training_curves.json`. Quick sanity check before downloading.

In [ ]:
import json
with open('bge-au-housing-v1/training_curves.json', 'r', encoding='utf-8') as f:
    curves = json.load(f)['curves']
for row in curves:
    print(f"epoch={row['epoch']} steps={row['steps']} MAP={row['map']:.4f} MRR@10={row['mrr_at_10']:.4f}")

## 6. Zip and download

The full SentenceTransformer dir is ~440 MB; zipping just packages it for transfer.

In [ ]:
!zip -qr bge-au-housing-v1.zip bge-au-housing-v1/
!ls -lh bge-au-housing-v1.zip

In [ ]:
from google.colab import files
files.download('bge-au-housing-v1.zip')

## Back on your laptop

```bash
mkdir -p models
unzip ~/Downloads/bge-au-housing-v1.zip -d models/
```

Then move on to **Task 2.11** (`reembed_finetuned.py`) to re-encode the 41,959 chunks with the fine-tuned model.